In [ ]:
# Project Name: Next-Out
# Description: Summarize segment data from many files single Excel file.
# Copyright (c) 2025 Justin Edenbaum, Never Gray
#
# This file is licensed under the MIT License.
# You may obtain a copy of the license at https://opensource.org/licenses/MIT

from pathlib import Path
import pandas as pd
import gzip
import pickle

In [ ]:
# Read in the NO file. You need to use the same version of Python that created the NO file.
# Next-Out Version 1.3.1 uses Python 3.13

def read_no_file(no_file_path):
    with gzip.open(no_file_path, 'rb') as file:
        no_file = pickle.load(file)
    data = no_file['data']
    output_meta_data = no_file['output_meta_data']
    return data, output_meta_data

In [ ]:
# Retreive segment data from a sepcific dataframe, for a specific column
# IF sub-segment is needed, refer to summary_output R02
def get_segment_data(data, segment, df_name, column_name, last_time_value=None):
    if last_time_value is None:
        last_time_value = data[df_name].index.get_level_values('Time').max()
    segment_data = data[df_name].loc[(last_time_value, segment), column_name]
    return segment_data

In [ ]:
# Create a flattened dataframe for multiple segments, multiple data, in multiple simulations
def summarize_segment_data(no_directory_path, segments_2_lookup, data_2_lookup):
    summary_data = []

    # Iterate over all files with a suffix of '.no' in the directory
    for no_file_path in no_directory_path.glob('*.no'):
        # Get the fire segment and airflow value
        print(no_file_path)
        data, output_meta_data = read_no_file(no_file_path)
        # For each segment, get the segment data requested
        for segment in segments_2_lookup:
            # Save the file name and segment number in a dictionary. 
            # Erase any previous dictionary data if this is the next segment.
            data_dictionary = {
                    'File Name': no_file_path.stem,
                    'Segment': segment,
            }
            # Add the segment data for each column requested in the data_2_lookup dictionary
            for column_name, df_name in data_2_lookup.items():
                segment_data = get_segment_data(data, segment, df_name, column_name)
                data_dictionary[column_name] = segment_data
            # Add the data dictionary to the summary data list. Each list is a row in the summary dataframe.
            summary_data.append(data_dictionary)
    # Convert summary data into a dataframe
    summary_df = pd.DataFrame(summary_data)
    return summary_df


In [ ]:
from pathlib import Path
# Enter the Directory where the NO files are located
no_directory_path = Path('C:\\Simulations\\PT10')
# Specify the segements to lookup in a list
segments_2_lookup = [800, 801, 802, 808, 809, 812, 816, 817, 820, 821, 824, 432, 439, 532, 539]
# Specify the data to lookup for each segment
# Form at is {"Column Name": 'DataFrame where column is located'}
data_2_lookup = {"Max_Velocity":'SA', 
                 "Min_Velocity":'SA'}
summary_df = summarize_segment_data(no_directory_path, segments_2_lookup, data_2_lookup)

In [ ]:
# Name of Excel file is based on the directory name
excel_name = f"{no_directory_path.name}_summary.xlsx"
# Save to excel, but don't include the index (0, 1, 2, ...) in the left column
summary_df.to_excel(no_directory_path / excel_name, index=False)